# 이전 4시간 구성 · 선택 복습
현재 필수 수업은 [교재 안내](../../README.md)를 따릅니다. 이 파일은 이전 상세 실습 보존본입니다.


# 관찰 기록으로 변화 요약과 질의응답

**목표:** 선택한 관찰 기록의 근거와 시각을 추적하며 요약합니다.

60분 수업: 그림·개념 설명 → 실행 전 예측 → 코드 실행 → 원문·기록 비교. 앞 세 교시는 휴식 10분 포함, 마지막 교시는 종합 평가 포함입니다.

[교재 안내](../../README.md) · [슬라이드 92~101쪽](https://docs.google.com/presentation/d/1IIy5LG5Yue5sJR5DOzmtW6jOuX0DDIZb3LsswTamjcw/edit) · [배포 PDF](../../../슬라이드/강의자료_2일차_v2.pdf#page=92)

**커널:** Python (Day2 LangChain). 설치는 개인 컨테이너 Terminal에서 `bash /workspace/04.LLM_v2/scripts/install_langchain.sh`를 실행합니다. 설치가 끝나면 커널을 선택하고 아래 환경 셀부터 실행하세요. 본문 코드는 실행 가능한 기준 구현이며 마지막에는 별도의 TODO가 있습니다. 코드 작성 여부와 수정할 파일을 TODO 안내에서 확인하세요. TODO는 수업 마지막 10~15분 동안 진행합니다.

In [ ]:
from pathlib import Path
import json, os, sys
ROOT=next((p for p in (Path.cwd(),*Path.cwd().parents) if (p/'04.LLM_v2').is_dir()),None)
if ROOT is None:raise RuntimeError('배포 폴더 안에서 노트북을 여세요.')
COURSE=ROOT/'05.VLM_v2'
sys.path.insert(0,str(COURSE))
from vlm_v2.core import *
print('Python:',sys.executable)
print('교재:',COURSE)

## 1. 요약 모델은 기록 텍스트를 읽습니다

카메라 프레임은 VLM이 먼저 설명합니다.

그 설명 기록을 텍스트 모델에 전달합니다.

요약 모델이 원본 영상을 다시 본 것은 아닙니다.

원래 관찰의 오류가 요약에 이어질 수 있습니다.

오전 PDF RAG와 마찬가지로 근거 자료를 입력에 포함합니다. 여기서는 시간 순서로 최근 기록을 선택하며 의미 검색을 사용하는 것은 아닙니다. gemma3:1b는 이미지를 보지 않고 텍스트 기록을 요약합니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

![요약 모델은 기록 텍스트를 읽습니다](../../assets/04_observation_summary-01.svg)

## 2. 시간창을 먼저 선택하고 오류를 제외합니다

최근 max_records개 행을 선택합니다.

그 안에서 status=ok인 관찰만 사용합니다.

오류·dry-run 행은 근거에서 제외합니다.

제외된 ID와 사용한 ID를 함께 기록합니다.

오류를 먼저 지우고 오래된 행으로 채우면 “최근 구간”의 의미가 달라집니다. 코드가 선택한 window_ids와 selected_ids를 구분해 확인하세요. 제한을 늘리면 입력 길이와 지연이 커집니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

![시간창을 먼저 선택하고 오류를 제외합니다](../../assets/04_observation_summary-02.svg)

## 3. 자신의 로그를 명시적으로 선택합니다

OBSERVATION_LOG에 실제 JSONL 경로를 넣습니다.

비어 있으면 제공한 모의 기록으로 연습합니다.

모의 기록은 실제 카메라 검증이 아닙니다.

실제 실습 완료에는 자기 로그를 사용합니다.

노트북은 카메라 없는 환경에서도 절차를 설명할 수 있도록 모의 데이터를 제공합니다. 화면과 결과에서 모의/실제를 구분합니다. 실제 로그의 원본 이미지가 존재하고 해시가 일치하는지 확인하세요.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

In [ ]:
import os
OBSERVATION_LOG=os.environ.get('OBSERVATION_LOG','')  # TODO: 실제 로그 경로
log=Path(OBSERVATION_LOG) if OBSERVATION_LOG else COURSE/'data/demo_observations.jsonl'
records=load_observations(log)
print('실제 선택 로그' if OBSERVATION_LOG else '모의 기록 연습 — 실제 카메라 아님',log)
for r in records: print(r['id'],r['captured_at_utc'],r['status'])

## 4. 기록에서 말할 수 있는 범위를 정합니다

각 시점에 기록된 내용을 요약할 수 있습니다.

두 기록의 기술이 다르다는 점을 비교할 수 있습니다.

미관찰 시간의 사건은 확정할 수 없습니다.

등장·이동·원인의 단정은 원본 확인이 필요합니다.

같은 장면에 모델이 다른 설명을 했을 수도 있습니다. 텍스트 차이가 실제 장면 변화인지 보려면 두 원본 프레임을 나란히 확인해야 합니다. 물체가 사라졌다는 설명이 가림이나 시야 변화 때문일 수도 있습니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

## 5. 시간 순서와 차이를 요약합니다

선택한 기록을 system 근거로 전달합니다.

answer·sources·needs_review를 요청합니다.

출처는 관찰 ID와 원문 인용입니다.

결과에 선택·제외 목록과 원본 연결을 남깁니다.

본문에는 영상 전체를 관찰한 것처럼 쓰지 않습니다. “기록 A에서는 …, 기록 B에서는 …로 기술되었다”처럼 범위를 드러내면 검토하기 쉽습니다. 모델이 이를 어기면 사람이 평가에서 표시합니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

In [ ]:
summary_model=make_model(num_predict=384)
result=summarize(summary_model,records,max_records=12)
print(result['status'],result['answer'],result['error'])
print('사용:',result['selected_ids'],'제외:',result['excluded_ids'])
print(result['log_path'])

## 6. 출처 ID가 실제 기록을 가리키는지 봅니다

제공되지 않은 ID는 거부합니다.

quote가 기록 원문에 포함되는지 검사합니다.

인용 존재와 의미 뒷받침은 다릅니다.

원본 이미지·해시로 돌아가 관찰을 확인합니다.

요약 결과가 “검증됐다”고 말할 때 무엇을 검사했는지 구체적으로 구분하세요. ID와 문자열 인용만 자동 검사합니다. 시간 변화나 인과 판단의 타당성은 자동으로 보장하지 않습니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

![출처 ID가 실제 기록을 가리키는지 봅니다](../../assets/04_observation_summary-06.svg)

## 7. 기록에 질문합니다

어느 시점에 사람이 기록됐는지 묻습니다.

시야 불명확 또는 실패한 구간을 확인합니다.

기록 밖 질문에는 보류하는지 봅니다.

각 질문은 선택한 기록에 독립적으로 질의합니다.

이 CLI는 사용자 대화를 기억하지 않고 선택한 관찰 기록을 매 질문 다시 제공합니다. “그럼 그때는?”처럼 생략하는 대신 대상과 시각을 적습니다. 대화 기억을 추가하는 확장은 오전 구현과 연결하는 선택 과제입니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

In [ ]:
qa=summarize(summary_model,records,'기록에서 사람이나 차량이 언급된 시점과 근거를 알려 주세요.')
print(qa['status'],qa['answer'],qa['error'])

## 8. 터미널에서 관찰 기록에 질의합니다

summary_chat.py에 자신의 JSONL을 지정합니다.

--question은 한 질문 후 종료합니다.

옵션이 없으면 질문을 반복 입력합니다.

/exit로 종료하고 각 요약 기록을 확인합니다.

화면의 경로 대신 앞 교시에서 생성된 정확한 경로를 넣습니다. 요청을 다시 실행해도 기존 관찰 로그를 덮어쓰지 않습니다. 요약 결과는 summary-실행ID 폴더에 저장됩니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

**개인 컨테이너 Terminal 명령** (Ollama 접속 명령은 Jetson 호스트):

```bash
cd /workspace
PY=Data/venvs/day2-langchain/bin/python
read -r -p "JSONL 경로: " OBSERVATION_LOG
"$PY" 05.VLM_v2/scripts/summary_chat.py \
  "$OBSERVATION_LOG"
```
카메라 실행 결과에 출력된 실제 JSONL의 절대 경로를 입력합니다. 노트북의 Python 변수는 터미널 셸 변수에 자동 전달되지 않습니다.


## 9. 종합 평가 · 오류가 어디에서 시작됐나요?

카메라 입력·프레임 선택·VLM 관찰을 구분합니다.

기록 선택·요약·출처 검사를 구분합니다.

틀린 요약은 원본 관찰까지 거슬러 확인합니다.

기록 없는 시간은 평가 한계로 남깁니다.

모델 출력을 후속 모델의 입력으로 사용하는 구조에서는 오류가 누적될 수 있습니다. 원본이 맞는지, 관찰이 맞는지, 요약이 기록에 충실한지 순서대로 확인하세요. 실제 동작 제어에 바로 연결하지 않습니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

In [ ]:
review={'log':str(log),'demo':not bool(OBSERVATION_LOG),'summary_log':result['log_path'],
        'observation_correct':None,'summary_supported':None,'unobserved_gap_noted':None,'notes':''}
path=new_run('review')/'review.json'
path.write_text(json.dumps(review,ensure_ascii=False,indent=2))
print(path)

## 10. 2일차 정리 · 입력 근거와 기록으로 설명합니다

오전: LangChain 챗봇·기억·PDF RAG.

오후: 이미지 질문·카메라 기록·변화 요약.

입력·모델·응답·시간·오류를 함께 남겼습니다.

형식 성공·사실 정답·검토 필요를 구분합니다.

이론 20분, 실습 25분, 종합 평가 15분입니다. 완료하려면 실제 카메라 또는 명시한 녹화 입력의 관찰 로그와 그 로그를 사용한 요약을 제출합니다. 모의 기록 실행은 준비 확인이며 실제 관찰 실습 완료가 아닙니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

## 마지막 TODO — 요약할 최근 관찰 기록 선택

**코드 작성: 있음 · 15분 · 수업 시간 안에 진행**

수정 파일: [practice/observation_window.py](../../practice/observation_window.py). JupyterLab 편집기에서 함수를 완성하고 저장한 뒤 아래 로딩 셀과 검사 셀을 다시 실행하세요. 노트북은 학생 파일을 덮어쓰지 않습니다.

**입력·출력 계약:** 최근 max_records개 행을 먼저 선택하세요. 그 안에서 status=ok이고 raw가 공백 아닌 문자열인 행만 usable에 넣고 나머지는 excluded에 넣으세요. {"window":...,"usable":...,"excluded":...} dict를 반환합니다. 순서와 원본 목록·행을 보존하며 반환값은 깊은 복사입니다. max_records는 1 이상의 정수이고 bool은 허용하지 않습니다.

**힌트:** 먼저 [-max_records:]로 자른 뒤 정상 행을 거르세요. 먼저 정상 행을 거르면 오래된 기록이 끼어들 수 있습니다.

미완성이면 마지막 검사에서 NotImplementedError가 나는 것이 정상입니다. 검사 통과 후 아래 적용 예제의 중간값을 설명하세요. 이 예제는 학생 함수를 직접 사용하며 본문의 기준 클래스/카메라 프로그램을 자동 교체하지 않습니다.

In [ ]:
import importlib.util, inspect, os
runner_path=COURSE/'practice/runner.py'
spec=importlib.util.spec_from_file_location('day2_todo_runner',runner_path)
runner=importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
VARIANT=os.environ.get('DAY2_TODO_VARIANT','exercise')
task=runner.load('observation_window',VARIANT)
print('선택 구현:',VARIANT)
print(inspect.getsource(task.select_window))

In [ ]:
print(runner.check('observation_window',task))

from vlm_v2.core import load_observations
import os
selected_path=os.environ.get('OBSERVATION_LOG','')
log=Path(selected_path) if selected_path else COURSE/'data/demo_observations.jsonl'
rows=load_observations(log)
selection=task.select_window(rows,2)
print('실제 로그' if selected_path else '제공된 모의 로그',log)
for key in ['window','usable','excluded']:print(key,[r['id'] for r in selection[key]])
context='\n'.join(f"[{r['id']}] {r['captured_at_utc']} {r['raw']}" for r in selection['usable'])
print('요약에 전달할 문맥:',context)

### TODO 완료와 참고 정답

- [ ] 함수를 직접 작성하고 검사 셀을 통과했습니다.
- [ ] 경계 입력 한 개를 추가해 예상 결과를 설명했습니다.
- [ ] 적용 예제의 입력과 반환값을 설명했습니다.

강사가 정한 직접 시도 시간 뒤 [참고 정답](../../practice/solutions/observation_window.py)을 비교하세요. 참고 정답을 실행했다고 학생 파일이 완성되는 것은 아닙니다.

**해설:** 최근 시간창의 오류를 지우고 오래된 성공 행으로 채우면 요약 범위가 바뀝니다. 제외된 ID도 남기면 요약이 무엇을 보지 못했는지 설명할 수 있습니다.

추가 모델 요청 없이 함수와 데이터 흐름을 검사합니다. 실제 모델·카메라 결과 평가는 본문 실습에서 수행합니다.

## 완료 확인

실제 입력과 출력 기록을 확인하고, 형식·근거·지연을 구분해 설명합니다. 오류는 숨기지 않고 원문과 함께 남깁니다. 추가 실험은 한 조건만 바꿔 비교합니다.